# 28 · Supplementary Figures 1 to 7, and two fixes to notebook 27

Draws the supplementary figures the review found missing, in the style of notebook 27, and redraws two figures from notebook 27 with small fixes. No model is run; everything comes from saved predictions or saved aggregate results.

| Figure | Content | Source |
|---|---|---|
| Supp Fig 1 | Retina: benefit at every referral share (5% to 90%), all pairs and the strongest partner, EyePACS and DDR, PDR | saved predictions; strongest partner = highest EyePACS validation AUC among other architectures (no test labels) |
| Supp Fig 2 | Skin: benefit at every referral share (20% to 90%, as in the dated plan of 27 Sep), all pairs and the strongest partner | saved predictions; strongest partner = highest internal validation AUC (`training_record.json`) |
| Supp Fig 3 | Retina: share of misses rescued by grade, EyePACS, DDR and BRSET | saved predictions (as Figure 4) |
| Supp Fig 4 | Retina: shared misses by starting weights and architecture | `starting_weights_part2_results.json` |
| Supp Fig 5 | Retina: fingerprint similarity against shared severe misses, adjusted for strength, with the corrected model bootstrap | `ballistics_full/pair_table.csv`, `retina_revisions_c1_a.json` (R7) |
| Supp Fig 6 | Choosing a partner from a random local sample | `partner_choice_calibration_results.json` |
| Supp Fig 7 | Choosing a partner from an enriched local sample, split by patient and by image | `retina_revisions_c1_b.json` (R5) |
| Figure 5, Supp Fig 13 | Fixes: BRSET kappa label rounded half up (63%, as in the text); Supp Fig 13 footnote moved clear of the axis labels | as notebook 27 |

Supp Fig 8 (HIBA by image type) is unchanged. Older versions are renamed `SUPERSEDED_` in `05_figures/manuscript_final/`. The last cells print the numbers each figure shows next to the numbers quoted in the manuscript, and write `supp_figures_c1_data.json`.

BRSET is credentialed data: it is read only inside this runtime, and only figures and aggregate numbers are written. CPU, High-RAM; about 25 to 40 minutes.

In [1]:
# Cell 1: load both domains
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
OUT = f'{BASE}/05_figures/manuscript_final'; os.makedirs(OUT, exist_ok=True)
def pick(df, c): return next(x for x in c if x in df.columns)
def tag(k): return '' if k == 0 else f'_seed{k}'
PAPER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
         'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
CK = {'densenet121': 'densenet', 'efficientnet_b3': 'effnet', 'resnet50': 'resnet', 'vit_base': 'vit', 'retfound': 'retfound'}
ck_of = lambda v: next(c for p, c in CK.items() if v.startswith(p))
def load(path, ids, col=None):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = col or pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
DOM = {}
fr = pd.read_csv(f'{BASE}/01_splits/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
dd = pd.read_csv(f'{BASE}/01_splits/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
for site, df, tpl, f in (('EyePACS', fr, 'eyepacs_predictions_fresh_{m}_v2.csv', 0.25), ('DDR', dd, 'ddr_predictions_{m}_v2.csv', None)):
    ids = df.image_id.to_numpy(); g = df[pick(df, ['level','grade'])].to_numpy().astype(int)
    M = {f'{v}{tag(k)}': {'s': load(f'{BASE}/03_predictions/' + tpl.format(m=f'{v}{tag(k)}'), ids), 'ckpt': ck_of(v), 'variant': v} for v in PAPER for k in (0, 1, 2)}
    DOM[site] = {'domain': 'retina', 'models': M, 'pos': g >= 2, 'classes': {'moderate': g == 2, 'severe NPDR': g == 3, 'PDR': g == 4},
                 'f': f if f else 1.31 * float((dd.grade[dd.half == 'A'] >= 2).mean())}
CLASSES8 = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True); ids = D.image_id.to_numpy()
    M = {}
    for s in ('HAM10000', 'BCN20000'):
        for a in ('resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2'):
            for k in (0, 1): M[f'{s}_{a}_seed{k}'] = {'s': load(f'{DERM}/predictions/{es}__{s}_{a}_seed{k}.csv', ids, 'p_malignant'), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
    mal = D.dx.isin(['MEL', 'BCC', 'SCC']).to_numpy()
    DOM[es] = {'domain': 'dermatology', 'models': M, 'pos': mal, 'classes': {'BCC': (D.dx == 'BCC').to_numpy(), 'SCC': (D.dx == 'SCC').to_numpy(), 'melanoma': (D.dx == 'MEL').to_numpy()},
               'f': 1.31 * mal.mean()}
# BRSET (credentialed data: stays in this runtime; only figures and aggregate numbers are written)
import hashlib
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
assert sha(f'{BR}/label_brset.csv') == json.load(open(f'{BR}/brset_access_record.json'))['label_file_sha256']
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str); gB = pd.to_numeric(L['DR_ICDR'], errors='coerce')
L = L[gB.isin([0, 1, 2, 3, 4])].copy(); L['grade'] = gB[gB.isin([0, 1, 2, 3, 4])].astype(int)
L['image_id'] = L['image_id'].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True); L = L.sort_values('image_id').reset_index(drop=True)
M = {}
for s in ('eyepacs', 'ddr'):
    for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'):
        for k in (0, 1, 2):
            M[f'{s}_{a}{tag(k)}'] = {'s': load(f'{BASE}/03_predictions/brset_predictions_{s}_{a}{tag(k)}_v2.csv', L.image_id.to_numpy()), 'ckpt': a, 'variant': f'{s}_{a}', 'source': s}
gb = L.grade.to_numpy()
DOM['BRSET'] = {'domain': 'retina', 'models': M, 'pos': gb >= 2, 'classes': {'moderate': gb == 2, 'severe NPDR': gb == 3, 'PDR': gb == 4}, 'f': 1.31 * float((gb >= 2).mean())}
for k, v in DOM.items(): print(k, len(v['models']), 'models, f =', round(v['f'], 3), {c: int(m.sum()) for c, m in v['classes'].items()})

Mounted at /content/drive
EyePACS 33 models, f = 0.25 {'moderate': 2373, 'severe NPDR': 384, 'PDR': 328}
DDR 33 models, f = 0.555 {'moderate': 4477, 'severe NPDR': 236, 'PDR': 913}
HIBA 24 models, f = 0.609 {'BCC': 340, 'SCC': 158, 'melanoma': 253}
PAD 24 models, f = 0.62 {'BCC': 839, 'SCC': 190, 'melanoma': 52}
BRSET 24 models, f = 0.074 {'moderate': 451, 'severe NPDR': 78, 'PDR': 396}


In [2]:
# Cell 2: per ordered pair and class: partner's own miss rate, union rescue share, matched-volume rescue share; pair relationship and referral agreement
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def relation(site, A, B):
    if 'source' not in A:
        if A['ckpt'] != B['ckpt']: return 'different design'
        return 'seed sibling' if A['variant'] == B['variant'] else 'variant sibling'
    if A['variant'] == B['variant']: return 'seed sibling'
    if A['ckpt'] == B['ckpt']: return 'same checkpoint, different data'
    return 'different design, same data' if A['source'] == B['source'] else 'different design, different data'
ROWS = []
for site, S in DOM.items():
    M = S['models']; names = list(M); n = len(S['pos'])
    R = {m: refer(M[m]['s'], S['f']) for m in names}; RK = {m: rankdata(M[m]['s']) / n for m in names}
    for a, b in itertools.permutations(names, 2):
        e = refer(np.maximum(RK[a], RK[b]), S['f']); rel = relation(site, M[a], M[b]); kap = kappa(R[a], R[b])
        for c, y in S['classes'].items():
            ma, mb = y & ~R[a], y & ~R[b]; na = ma.sum()
            if na == 0: continue
            ROWS.append({'site': site, 'domain': S['domain'], 'a': a, 'b': b, 'relationship': rel, 'kappa': kap, 'class': c,
                         'partner_miss_rate': mb.sum() / y.sum(), 'anchor_miss_rate': na / y.sum(),
                         'union_rescue_share': (ma & ~mb).sum() / na, 'mv_rescue_share': (na - (y & ~e).sum()) / na,
                         'lift': ((ma & mb).sum() / y.sum()) / ((na / y.sum()) * (mb.sum() / y.sum())) if mb.sum() else np.nan})
PR = pd.DataFrame(ROWS); print(PR.groupby(['site', 'class']).size().to_string())
# rank correlation for every unordered pair (Figure 5, lower row)
PK = PR.drop_duplicates(['site', 'a', 'b'])[['site', 'domain', 'a', 'b', 'relationship', 'kappa']]; PK = PK[PK.a < PK.b].copy()
RANKS = {}
for site, S in DOM.items():
    n = len(S['pos']); RK = {m: rankdata(S['models'][m]['s']) / n for m in S['models']}
    RANKS[site] = {(a, b): float(np.corrcoef(RK[a], RK[b])[0, 1]) for a, b in itertools.combinations(sorted(S['models']), 2)}
PK['rank_corr'] = [RANKS[r.site][(r.a, r.b)] for r in PK.itertuples()]
CLASSN = {(site, c): int(m.sum()) for site, S in DOM.items() for c, m in S['classes'].items()}
print('pairs:', PK.groupby('site').size().to_dict())

site     class      
BRSET    PDR             552
         moderate        552
         severe NPDR     529
DDR      PDR            1056
         moderate       1056
         severe NPDR     960
EyePACS  PDR            1056
         moderate       1056
         severe NPDR    1056
HIBA     BCC             552
         SCC             552
         melanoma        552
PAD      BCC             552
         SCC             552
         melanoma        552
pairs: {'BRSET': 276, 'DDR': 528, 'EyePACS': 528, 'HIBA': 276, 'PAD': 276}


In [3]:
# Shared style
import shutil
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#333333', 'axes.linewidth': 0.8, 'xtick.color': '#333333', 'ytick.color': '#333333',
                     'axes.labelcolor': '#222222', 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.labelsize': 8.5, 'legend.frameon': False})
BLUE, SKY, ORANGE, VERM, PINK, GREY, INK, GREEN = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#8C8C8C', '#222222', '#009E73'
DOMCOL = {'retina': BLUE, 'dermatology': VERM}
SITES = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']
SNAME = {'EyePACS': 'EyePACS (US)', 'DDR': 'DDR (China)', 'BRSET': 'BRSET (Brazil)', 'HIBA': 'HIBA (Argentina)', 'PAD': 'PAD-UFES-20 (Brazil)'}
SDOM = {'EyePACS': 'retina', 'DDR': 'retina', 'BRSET': 'retina', 'HIBA': 'dermatology', 'PAD': 'dermatology'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
LS = {'EyePACS': '-', 'DDR': '--', 'BRSET': ':', 'HIBA': '-', 'PAD': '--'}
CNAME = {'moderate': 'Moderate NPDR', 'severe NPDR': 'Severe NPDR', 'PDR': 'PDR', 'BCC': 'BCC', 'SCC': 'SCC', 'melanoma': 'Melanoma'}
# relationship of two systems: one display name and one colour, used in Figures 3, 5 and 6
RELMAP = {'different design, different data': 'Different design, different data', 'different design, same data': 'Different design, same data',
          'different design': 'Different design, same data', 'same checkpoint, different data': 'Same design, different data',
          'variant sibling': 'Variant sibling (same design and data)', 'seed sibling': 'Seed sibling (repeat run)'}
RELS = ['Different design, different data', 'Different design, same data', 'Same design, different data', 'Variant sibling (same design and data)', 'Seed sibling (repeat run)']
RELCOL = dict(zip(RELS, [BLUE, SKY, ORANGE, PINK, VERM]))
def ci_txt(m, lo, hi, d=2, pct=False):
    u = '%' if pct else ''
    return f'{m:.{d}f}{u} ({lo:.{d}f} to {hi:.{d}f})'
def panel(ax, letter, text, sub=None):
    ax.set_title(f'{letter}  {text}', loc='left', fontsize=9.5, weight='bold', pad=14 if sub else 6)
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color='#444444', va='bottom')
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M')
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = f'{OUT}/{name}.{ext}'
        if os.path.exists(p):
            dest = f'{OUT}/SUPERSEDED_{name}.{ext}'
            if os.path.exists(dest): dest = f'{OUT}/SUPERSEDED_{name}_{STAMP}.{ext}'
            shutil.move(p, dest)
    fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
PK['rel'] = PK.relationship.map(RELMAP); PR['rel'] = PR.relationship.map(RELMAP)
DR = json.load(open(f'{DERM}/derm_results_v1.json'))
RESULTS = f'{BASE}/04_results'
REV = f'{RESULTS}/revisions_c1'
REG = json.load(open(f'{REV}/regret_c1.json'))['results']; ADA = json.load(open(f'{REV}/adaptive_rule_c1.json'))['results']
DEP = json.load(open(f'{REV}/deployment_c1.json')); POOL = json.load(open(f'{REV}/pooled_vs_pairing_c1.json'))
PAIRS = pd.read_csv(f'{REV}/regret_c1_pairs.csv'); BRR = json.load(open(f'{RESULTS}/brset_results_v6.json'))
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, head, body, ec, fc, hs=10, bs=8.3, lw=1.4, ital=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.004,rounding_size=0.01', ec=ec, fc=fc, lw=lw, transform=ax.transAxes))
    if head: ax.text(x + w / 2, y + h - 0.016, head, ha='center', va='top', fontsize=hs, weight='bold', color=INK, transform=ax.transAxes)
    if body: ax.text(x + w / 2, (y + h - 0.05) if head else (y + h / 2), body, ha='center', va='top' if head else 'center', fontsize=bs, color=INK,
                     linespacing=1.35, style='italic' if ital else 'normal', transform=ax.transAxes)
def arrow(ax, x0, y0, x1, y1, col='#666666'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), xycoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color=col, lw=1.3, shrinkA=0, shrinkB=0))
def gtag(ax, x, y, t='confirmatory, dated plan'):
    ax.text(x, y, t, ha='center', va='center', fontsize=7.4, color='white', weight='bold', transform=ax.transAxes, bbox=dict(boxstyle='round,pad=0.3', fc=GREEN, ec='none'))
def blank(ax): ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
LBL = {'EyePACS_severe': 'EyePACS, severe DR', 'DDR_severe': 'DDR, severe DR', 'DDR_pdr': 'DDR, PDR', 'BRSET_severe': 'BRSET, severe DR',
       'BRSET_mixed_severe': 'BRSET, mixed sources, severe DR', 'HIBA_mel': 'HIBA, melanoma', 'HIBA_mal': 'HIBA, any skin cancer',
       'PAD_mel': 'PAD-UFES-20, melanoma', 'PAD_mal': 'PAD-UFES-20, any skin cancer'}
LBL_DEP = {**LBL, 'DDR_pdr': 'DDR, PDR'}
KC = {'EyePACS_severe': '#0072B2', 'DDR_severe': '#56B4E9', 'DDR_pdr': '#009E73', 'BRSET_severe': '#003A70', 'BRSET_mixed_severe': '#8C8C8C',
      'HIBA_mel': '#D55E00', 'HIBA_mal': '#E69F00', 'PAD_mel': '#CC79A7', 'PAD_mal': '#8B4513'}
KM = lambda k: '^' if k.split('_')[0] in ('HIBA', 'PAD') else 'o'
def p0(x): return f'{int(np.floor(100 * x + 0.5))}%'   # percentages rounded half up, as in the text


In [4]:
# Extra inputs: patient groups, validation AUCs, result files
import glob
from sklearn.metrics import roc_auc_score
def find(name):
    for r in (RESULTS, REV, BASE, DERM):
        if os.path.exists(f'{r}/{name}'): return f'{r}/{name}'
    for r in (BASE, DERM):
        g = sorted(glob.glob(f'{r}/**/{name}', recursive=True))
        if g: return g[0]
    raise FileNotFoundError(name)
ep = pd.read_csv(f'{BASE}/03_predictions/eyepacs_predictions_fresh_densenet121_binary_v2.csv', dtype={'image_id': str, 'patient_id': str}).drop_duplicates('image_id').set_index('image_id')
GROUPS = {'EyePACS': ep['patient_id'].reindex(fr.image_id).astype(str).to_numpy(),
          'DDR': dd[pick(dd, ['group', 'filename_group', 'group_id'])].astype(str).to_numpy()}
assert not pd.isna(ep['patient_id'].reindex(fr.image_id)).any()
for es in ('HIBA', 'PAD'):
    D = pd.read_csv(f'{DERM}/manifest_{es}.csv', dtype={'image_id': str, 'patient_id': str}).sort_values('image_id')
    D = D[~D.duplicated('raw_sha256', keep='first') & D.dx.isin(CLASSES8)].reset_index(drop=True); GROUPS[es] = D.patient_id.astype(str).fillna('none').to_numpy()
    assert len(GROUPS[es]) == len(DOM[es]['pos'])
VAL = {}
for m in DOM['EyePACS']['models']:
    fn = f'{BASE}/03_predictions/eyepacs_predictions_val_{m}_v2.csv'
    d_ = pd.read_csv(fn); VAL[m] = float(roc_auc_score(d_[pick(d_, ['true_label', 'label'])], d_[pick(d_, ['predicted_probability', 'probability', 'prob'])]))
TR = json.load(open(find('training_record.json')))
for m in DOM['HIBA']['models']: VAL[m] = float(TR[m]['val_malignant_auc'])
print('validation AUCs:', len(VAL), '| EyePACS best', max((v, k) for k, v in VAL.items() if k in DOM['EyePACS']['models']))
def boot_w(groups, n_boot, seed):
    codes = pd.factorize(groups)[0]; G = codes.max() + 1; rng_ = np.random.default_rng(seed)
    W = np.empty((n_boot + 1, len(codes)), np.float32); W[0] = 1.0
    for b in range(1, n_boot + 1): W[b] = np.bincount(rng_.integers(0, G, G), minlength=G)[codes]
    return W
def pctl(x): x = np.asarray(x, float); return [float(x[0]), float(np.nanpercentile(x[1:], 2.5)), float(np.nanpercentile(x[1:], 97.5))]
CHECK = {}   # numbers shown in the figures next to the manuscript's

validation AUCs: 57 | EyePACS best (0.8826127986962535, 'densenet121_5class_seed2')


In [5]:
# Referral-share sweeps (Supplementary Figures 1 and 2): relative reduction in missed cases from the first system to the pair, pooled over pairs
def sweep(site, y, pairs, grid, B=1000, seed=20261030):
    S = DOM[site]; M = S['models']; n = len(y); W = boot_w(GROUPS[site], B, seed)
    used = sorted({m for p in pairs for m in p}); RK = {m: rankdata(M[m]['s']) / n for m in used}
    out = []
    for f in grid:
        R = {m: refer(M[m]['s'], f) for m in used}; A = np.zeros(n, np.float32); E = np.zeros(n, np.float32)
        for a, b in pairs:
            A += y & ~R[a]; E += y & ~refer(np.maximum(RK[a], RK[b]) + 1e-9 * np.minimum(RK[a], RK[b]), f)
        wa, we = W @ A, W @ E
        with np.errstate(divide='ignore', invalid='ignore'): rr = np.where(wa > 0, 1 - we / wa, np.nan)
        out.append(pctl(rr) + [float(A.sum() / len(pairs))])   # last column: misses per first system, full sample
    return np.array(out)
def strongest(site, m):
    M = DOM[site]['models']; c = [x for x in M if M[x]['ckpt'] != M[m]['ckpt']]; return max(c, key=lambda x: VAL[x])
SW = {}
GRID_R = np.round(np.arange(0.05, 0.901, 0.05), 2); GRID_S = np.round(np.arange(0.20, 0.901, 0.05), 2)
for site in ('EyePACS', 'DDR'):
    M = DOM[site]['models']; y = DOM[site]['classes']['PDR']
    allp = [(a, b) for a, b in itertools.permutations(M, 2) if M[a]['variant'] != M[b]['variant']]
    best = [(a, strongest(site, a)) for a in M]
    SW[(site, 'PDR')] = {'all pairs': sweep(site, y, allp, GRID_R), 'strongest partner': sweep(site, y, best, GRID_R)}
    print(site, 'PDR: strongest partner', ' '.join(f'{f:.2f}:{100 * v[0]:.0f}%' for f, v in zip(GRID_R, SW[(site, 'PDR')]['strongest partner'])))
for site, oc in (('HIBA', 'melanoma'), ('HIBA', 'any skin cancer'), ('PAD', 'any skin cancer')):
    M = DOM[site]['models']; y = DOM[site]['classes']['melanoma'] if oc == 'melanoma' else DOM[site]['pos']
    allp = [(a, b) for a, b in itertools.permutations(M, 2) if M[a]['ckpt'] != M[b]['ckpt']]
    best = [(a, strongest(site, a)) for a in M]
    SW[(site, oc)] = {'all pairs': sweep(site, y, allp, GRID_S), 'strongest partner': sweep(site, y, best, GRID_S)}
    print(site, oc, {k: f'{100 * v[:, 0].min():.1f}% to {100 * v[:, 0].max():.1f}%; interval above 0 at {(v[:, 1] > 0).sum()} of {len(v)} shares' for k, v in SW[(site, oc)].items()})
r_ = SW[('DDR', 'PDR')]['strongest partner']; r_ = r_[r_[:, 3] >= 10, 0]; CHECK['SuppFig1 DDR strongest partner range'] = {'figure': f'{100 * r_.min():.0f}% to {100 * r_.max():.0f}%', 'manuscript': '26% to 85%'}
CHECK['SuppFig2 skin all pairs'] = {'figure': {f'{s} {o}': f"{100 * SW[(s, o)]['all pairs'][:, 0].min():.1f}% to {100 * SW[(s, o)]['all pairs'][:, 0].max():.1f}%" for s, o in (('HIBA', 'melanoma'), ('HIBA', 'any skin cancer'), ('PAD', 'any skin cancer'))},
                                   'derm_referral_share_results (27 Sep)': 'HIBA melanoma 4.3% to 43.5%; HIBA any 0.9% to 56.9%; PAD any 0.6% to 27.6%'}

EyePACS PDR: strongest partner 0.05:11% 0.10:23% 0.15:30% 0.20:37% 0.25:50% 0.30:51% 0.35:54% 0.40:54% 0.45:59% 0.50:61% 0.55:65% 0.60:58% 0.65:63% 0.70:58% 0.75:52% 0.80:56% 0.85:86% 0.90:100%
DDR PDR: strongest partner 0.05:2% 0.10:4% 0.15:5% 0.20:7% 0.25:9% 0.30:13% 0.35:14% 0.40:20% 0.45:28% 0.50:32% 0.55:43% 0.60:48% 0.65:49% 0.70:51% 0.75:44% 0.80:40% 0.85:42% 0.90:36%
HIBA melanoma {'all pairs': '4.3% to 43.5%; interval above 0 at 15 of 15 shares', 'strongest partner': '1.5% to 50.0%; interval above 0 at 12 of 15 shares'}
HIBA any skin cancer {'all pairs': '0.9% to 56.8%; interval above 0 at 14 of 15 shares', 'strongest partner': '1.8% to 59.6%; interval above 0 at 14 of 15 shares'}
PAD any skin cancer {'all pairs': '0.6% to 27.7%; interval above 0 at 14 of 15 shares', 'strongest partner': '2.6% to 33.8%; interval above 0 at 15 of 15 shares'}


In [6]:
# Supplementary Figures 1 and 2
def sweep_fig(keys, grid, name, ttl, share_of):
    fig, axes = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 4.4))
    for ax, key, lt in zip(np.atleast_1d(axes), keys, 'abcd'):
        site, oc = key
        for lab, col, mk in (('all pairs', GREY, 'o'), ('strongest partner', DOMCOL[SDOM[site]], 'D')):
            v = 100 * SW[key][lab][:, :3].copy(); v[SW[key][lab][:, 3] < 10] = np.nan   # shares where each system misses fewer than 10 cases are not shown
            ax.fill_between(grid * 100, v[:, 1], v[:, 2], color=col, alpha=0.18, lw=0); ax.plot(grid * 100, v[:, 0], marker=mk, ms=4, color=col, lw=1.8, label=lab)
        ax.axhline(0, color='#333333', lw=0.8); ax.axvline(100 * share_of[site], color='#555555', lw=0.9, ls=':')
        ax.text(100 * share_of[site], 1.0, ' share used in the main analysis', transform=ax.get_xaxis_transform(), fontsize=7, color='#555555', va='top')
        ax.set_xlabel('Referral share (%)'); ax.set_ylabel('Reduction in missed cases vs the first system (%)'); panel(ax, lt, f'{SNAME[site]}: {oc}')
    np.atleast_1d(axes)[0].legend(fontsize=7.5, loc='lower right')
    fig.text(0.01, -0.02, 'Shares at which each system missed fewer than 10 cases are not shown.', fontsize=7.8, color='#444444')
    title(fig, ttl); fig.tight_layout(); save(fig, name)
SHARE = {s: DOM[s]['f'] for s in DOM}
sweep_fig([('EyePACS', 'PDR'), ('DDR', 'PDR')], GRID_R, 'SuppFig1_retina_referral_shares',
          'Supplementary Figure 1. Retina: benefit at every referral share (PDR; 95% intervals resample patients or filename groups)', SHARE)
sweep_fig([('HIBA', 'melanoma'), ('HIBA', 'any skin cancer'), ('PAD', 'any skin cancer')], GRID_S, 'SuppFig2_skin_referral_shares',
          'Supplementary Figure 2. Skin: benefit at every referral share (95% intervals resample patients)', SHARE)

saved SuppFig1_retina_referral_shares
saved SuppFig2_skin_referral_shares


In [7]:
# Supplementary Figure 3: share of misses rescued by retinopathy grade (pairs of different design)
GR = ['moderate', 'severe NPDR', 'PDR']
PR['anchor_misses'] = PR.anchor_miss_rate * PR.apply(lambda r: CLASSN[(r.site, r['class'])], axis=1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True); F3S = {}
for ax, site, lt in zip(axes, ('EyePACS', 'DDR', 'BRSET'), 'abc'):
    s = PR[(PR.site == site) & PR.relationship.str.startswith('different design')]
    g = s.groupby('class').agg(mv=('mv_rescue_share', 'median'), un=('union_rescue_share', 'median'), miss=('anchor_miss_rate', 'median'), nmiss=('anchor_misses', 'median')).reindex(GR)
    x = np.arange(len(GR))
    ax.bar(x - 0.19, 100 * g.mv, 0.36, color=BLUE, label='at the same referral volume')
    ax.bar(x + 0.19, 100 * g.un, 0.36, color=SKY, label='referring either system (union)')
    for i, r in enumerate(g.itertuples()):
        ax.text(i, max(100 * r.mv, 100 * r.un) + 2, f'miss rate {100 * r.miss:.0f}%' + ('\nfewer than 10 misses' if r.nmiss < 10 else ''), ha='center', fontsize=7.2, color='#444444')
    ax.set_xticks(x); ax.set_xticklabels([CNAME[c] for c in GR]); ax.set_ylim(0, 105)
    if lt == 'a': ax.set_ylabel("Median % of the first system's misses rescued")
    panel(ax, lt, SNAME[site], f"{s[['a', 'b']].drop_duplicates().shape[0]} ordered pairs")
    F3S[site] = g.round(4).to_dict(orient='index')
axes[0].legend(fontsize=7.5, loc='upper left')
title(fig, 'Supplementary Figure 3. Retina: share of misses rescued by grade (median over pairs of different design; per-system miss rate above each pair of bars)')
fig.tight_layout(); save(fig, 'SuppFig3_rescue_by_grade')
b5 = BRR['B5_miss_rate_by_grade']; CHECK['SuppFig3 BRSET matched-volume rescue by grade'] = {'figure': {c: round(F3S['BRSET'][c]['mv'], 3) for c in GR}, 'brset_results_v6 B5': {k: v['median_rescue_share_matched_volume'] for k, v in b5.items()}}

saved SuppFig3_rescue_by_grade


In [8]:
# Supplementary Figure 4: starting weights versus architecture (EyePACS test set; notebook STARTING_WEIGHTS_part2)
SWR = json.load(open(find('starting_weights_part2_results.json')))
TY = [('seed_siblings', 'Same design and weights\n(repeat runs)', VERM), ('same_arch_diff_init_vit', 'ViT-Base, different\npretrained weights', ORANGE),
      ('same_arch_diff_init_resnet', 'ResNet50, different\npretrained weights', PINK), ('different_architecture', 'Different architecture', BLUE)]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9))
for ax, (oc, ttl), lt in zip(axes, (('referable', 'Referable DR'), ('grade3plus', 'Severe DR (grade 3 or 4)'), ('pdr', 'PDR')), 'abc'):
    r = SWR['co_miss_ratio'][oc]; hi_all = max(r[t]['ci95'][1] for t, *_ in TY)
    for i, (t, lab, col) in enumerate(TY):
        m, (lo, hi) = r[t]['mean'], r[t]['ci95']
        ax.plot([lo, hi], [i, i], color=col, lw=2.2, solid_capstyle='round'); ax.plot(m, i, 'o', color=col, ms=7, mec='white', zorder=3)
        ax.text(hi + 0.03 * hi_all, i, ci_txt(m, lo, hi), va='center', fontsize=7.8)
    ax.set_yticks(range(len(TY))); ax.set_yticklabels([t[1] for t in TY] if lt == 'a' else [''] * len(TY)); ax.set_ylim(len(TY) - 0.5, -0.6)
    ax.axvline(1, color='#555555', lw=0.8, ls=':'); ax.set_xlim(0.8, hi_all * 1.45); ax.set_xlabel('Shared-miss ratio (1 = independent errors)'); panel(ax, lt, ttl)
title(fig, 'Supplementary Figure 4. Retina: changing pretrained starting weights reduces shared misses about as much as changing architecture (EyePACS test set; 95% intervals)')
fig.tight_layout(); save(fig, 'SuppFig4_starting_weights')
CHECK['SuppFig4 severe DR'] = {t: SWR['co_miss_ratio']['grade3plus'][t]['mean'] for t in ('seed_siblings', 'same_arch_diff_init_pooled', 'different_architecture')}

saved SuppFig4_starting_weights


In [9]:
# Supplementary Figure 5: fingerprint similarity against shared severe misses, adjusted for strength (22-model zoo, EyePACS test set)
FULL = f'{RESULTS}/ballistics_full'; PT = pd.read_csv(f'{FULL}/pair_table.csv'); D5 = PT.dropna(subset=['comiss_sev']).reset_index(drop=True)
R7 = json.load(open(find('retina_revisions_c1_a.json')))['R7_ballistics_corrected_bootstrap']['results']
def resid(df, col):
    Z_ = np.column_stack([np.ones(len(df)), rankdata(df.mean_sevmiss), rankdata(df.gap_sevmiss)]); v = rankdata(df[col])
    return v - Z_ @ np.linalg.lstsq(Z_, v, rcond=None)[0]
FLAB = {'sim_degradation': 'Degradation response', 'sim_frequency': 'Frequency response', 'sim_channel': 'Colour-channel response', 'sim_occlusion': 'Occlusion response',
        'cka_trained': 'Feature similarity (trained)', 'cka_pretrained': 'Feature similarity (pretrained)', 'sim_pretrained_shift': 'Shift from pretraining', 'sim_clean_scores': 'Clean-score similarity'}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={'width_ratios': [1, 1.15]})
ax = axes[0]; rx, ry = resid(D5, 'sim_degradation'), resid(D5, 'comiss_sev')
ax.scatter(rx, ry, s=14, color=BLUE, alpha=0.55, lw=0); b_ = np.polyfit(rx, ry, 1); xx = np.linspace(rx.min(), rx.max(), 50); ax.plot(xx, np.polyval(b_, xx), color=INK, lw=1.2)
ax.axhline(0, color='#999999', lw=0.6); ax.axvline(0, color='#999999', lw=0.6)
ax.set_xlabel('Degradation-fingerprint similarity (rank, adjusted for strength)'); ax.set_ylabel('Shared severe misses (rank, adjusted for strength)')
r7 = R7['sim_degradation']; panel(ax, 'a', f'Each dot is one pair of {len(set(D5.a) | set(D5.b))} models',
                                  f"Partial rank correlation {r7['partial_rho_given_strength']:.2f} ({r7['model_bootstrap_ci95_with_multiplicity'][0]:.2f} to {r7['model_bootstrap_ci95_with_multiplicity'][1]:.2f})")
ax = axes[1]; ks = [k for k in FLAB if k in R7]; ks = sorted(ks, key=lambda k: -R7[k]['partial_rho_given_strength'])
for i, k in enumerate(ks):
    r = R7[k]; lo, hi = r['model_bootstrap_ci95_with_multiplicity']; col = BLUE if k == 'sim_degradation' else GREY
    ax.plot([lo, hi], [i, i], color=col, lw=2.2, solid_capstyle='round'); ax.plot(r['partial_rho_given_strength'], i, 'o', color=col, ms=7, mec='white', zorder=3)
    ax.text(max(hi, 0) + 0.03, i, f"{r['partial_rho_given_strength']:.2f} ({lo:.2f} to {hi:.2f})", va='center', fontsize=7.8)
ax.set_yticks(range(len(ks))); ax.set_yticklabels([FLAB[k] for k in ks]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlim(-0.4, 1.0)
ax.set_xlabel('Partial rank correlation with shared severe misses'); panel(ax, 'b', 'Every fingerprint', 'Model bootstrap keeping repeated draws (1,000 resamples); degradation pre-specified')
title(fig, 'Supplementary Figure 5. Retina: systems that respond alike to image degradation share more severe misses, beyond their strength')
fig.tight_layout(); save(fig, 'SuppFig5_fingerprints')
CHECK['SuppFig5 degradation'] = {'figure': r7['partial_rho_given_strength'], 'interval': r7['model_bootstrap_ci95_with_multiplicity'], 'manuscript': '0.41 (0.18 to 0.60)'}

saved SuppFig5_fingerprints


In [10]:
# Supplementary Figures 6 and 7: choosing a partner locally
PC = json.load(open(find('partner_choice_calibration_results.json')))['summary']
RULES6 = [('sample_auc', 'Highest AUC on the sample', BLUE, 'o'), ('sample_severe_misses', 'Fewest severe misses on the sample', SKY, 's'),
          ('consensus', 'Agrees most with the other candidates (no labels)', ORANGE, '^'), ('consensus_plus_degradation', 'Consensus plus degradation fingerprint (no labels)', PINK, 'D')]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for ax, site, lt in zip(axes, ('EyePACS', 'DDR'), 'ab'):
    ns = sorted(PC[site], key=int)
    for k, lab, col, mk in RULES6:
        if k not in PC[site][ns[0]]: continue
        ax.plot([int(n) for n in ns], [100 * PC[site][n][k]['share_of_true_strongest_advantage_over_random'] for n in ns], marker=mk, color=col, lw=1.8, ms=5, label=lab)
    ax.axhline(100, color='#333333', lw=0.8, ls=':'); ax.axhline(0, color='#333333', lw=0.8); ax.set_xscale('log'); ax.minorticks_off(); ax.set_xticks([int(n) for n in ns]); ax.set_xticklabels(ns)
    ax.set_xlabel('Random local sample (images)'); panel(ax, lt, SNAME[site], 'Images drawn at random, not by patient')
axes[0].set_ylabel('Share of the gain from the strongest partner (%)'); axes[0].legend(fontsize=7.2, loc='lower right')
title(fig, 'Supplementary Figure 6. Choosing a partner from a random local sample (0 = choosing at random; 100 = the strongest standalone partner)')
fig.tight_layout(); save(fig, 'SuppFig6_partner_random_sample')
R5 = json.load(open(find('retina_revisions_c1_b.json')))['R5_patient_disjoint_partner_choice']
EC = json.load(open(find('enriched_calibration_results.json')))['summary']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True); F7S = {}
for ax, site, lt in zip(axes, ('EyePACS', 'DDR'), 'ab'):
    ks = sorted(R5[site], key=int); x = [int(k) for k in ks]
    nz = lambda v: np.nan if v is None else 100 * v
    pd_s = [nz(R5[site][k]['share_of_strongest_partner_gain']) for k in ks]; pd_o = [nz(R5[site][k]['share_of_oracle_gain']) for k in ks]
    im_s = [100 * EC[site][k]['enriched_auc']['share_of_true_strongest_advantage'] for k in ks]; im_o = [100 * EC[site][k]['enriched_auc']['share_of_oracle_advantage'] for k in ks]
    ax.plot(x, pd_s, marker='o', color=BLUE, lw=2, label='split by patient: share of strongest-partner gain')
    ax.plot(x, pd_o, marker='o', color=BLUE, lw=1.4, ls='--', mfc='white', label='split by patient: share of best-possible-partner gain')
    ax.plot(x, im_s, marker='s', color=GREY, lw=1.6, label='split by image (earlier version): strongest partner')
    ax.plot(x, im_o, marker='s', color=GREY, lw=1.2, ls='--', mfc='white', label='split by image: best possible partner')
    for xi, k in zip(x, ks): ax.text(xi, -12, f"{R5[site][k]['mean_unique_patients_with_severe_drawn']:.0f}", ha='center', fontsize=7, color='#555555')
    ax.axhline(0, color='#333333', lw=0.8); ax.axhline(100, color='#333333', lw=0.8, ls=':'); ax.set_xscale('log'); ax.minorticks_off(); ax.set_xticks(x); ax.set_xticklabels([f'{k} + {k}' for k in ks]); ax.set_ylim(-18, 105)
    ax.set_xlabel('Enriched sample (severe cases + others)'); panel(ax, lt, SNAME[site], 'Numbers above the axis: patients (DDR: filename groups) with severe DR drawn')
    F7S[site] = {k: [round(a, 1), round(b, 1)] for k, a, b in zip(ks, pd_s, pd_o)}
axes[0].set_ylabel('Share of the gain over random choice (%)'); axes[0].legend(fontsize=7, loc='upper left')
title(fig, 'Supplementary Figure 7. Choosing a partner from an enriched local sample: highest AUC on the sample, scored on patients outside it')
fig.tight_layout(); save(fig, 'SuppFig7_partner_enriched_sample')
CHECK['SuppFig7 patient split, strongest / best possible'] = {'figure': F7S, 'manuscript': 'EyePACS 40/30, 53/39, 75/55, 79/56; DDR 19/17, 36/32, 49/42, 63/52'}

saved SuppFig6_partner_random_sample
saved SuppFig7_partner_enriched_sample


In [11]:
# Figure 5: related systems give themselves away (agreement by relationship, five populations; relative rule as in FIGURES_brset)
from sklearn.metrics import roc_auc_score
DET = {}
fig, axes = plt.subplots(2, 5, figsize=(17, 7.4), sharey=True)
rng = np.random.default_rng(0)
for row, (metric, lab) in enumerate((('kappa', 'Agreement on referral decisions (kappa)'), ('rank_corr', 'Agreement on score rankings (rank correlation)'))):
    for ax, site in zip(axes[row], SITES):
        s = PK[PK.site == site]; sib = s.relationship.str.contains('sibling'); dif = s.relationship.str.startswith('different design')
        thr = float(np.percentile(s[metric][dif], 95)); flag = float((s[metric][sib] > thr).mean())
        auc = float(roc_auc_score(np.r_[np.ones(sib.sum()), np.zeros(dif.sum())], np.r_[s[metric][sib], s[metric][dif]]))
        DET.setdefault(site, {})[metric] = {'auc_siblings_vs_different_design': round(auc, 3), 'relative_threshold': round(thr, 4), 'siblings_flagged': round(flag, 3)}
        for i, rl in enumerate(RELS):
            v = s[metric][s.rel == rl].to_numpy()
            if len(v) == 0:
                ax.text(0.02, i, 'not in this design', transform=ax.get_yaxis_transform(), va='center', fontsize=7, color='#9a9a9a'); continue
            bp = ax.boxplot([v], positions=[i], vert=False, widths=0.55, patch_artist=True, showfliers=False, whis=(2.5, 97.5),
                            medianprops=dict(color=INK, lw=1.4), whiskerprops=dict(color='#555555', lw=0.8), capprops=dict(color='#555555', lw=0.8),
                            boxprops=dict(facecolor=RELCOL[rl], alpha=0.35, edgecolor=RELCOL[rl], lw=1))
            jit = rng.uniform(-0.18, 0.18, len(v)) if len(v) > 1 else np.zeros(1)
            ax.scatter(v, i + jit, s=4 if len(v) > 200 else 7, color=RELCOL[rl], alpha=0.45 if len(v) > 200 else 0.8, lw=0, zorder=3, rasterized=True)
        ax.axvline(thr, color=GREEN, ls='-', lw=1.4)
        ax.set_yticks(range(len(RELS))); ax.set_yticklabels(RELS); ax.set_ylim(len(RELS) - 0.5, -0.6)
        ax.set_xlabel(lab, fontsize=8); ax.tick_params(axis='y', length=0)
        ax.set_title(f'{SNAME[site]}', loc='left', fontsize=9, weight='bold', pad=14)
        ax.text(0, 1.015, f'AUC {auc:.2f}; {p0(flag)} of siblings flagged', transform=ax.transAxes, fontsize=7.5, color='#444444', va='bottom')
for row, (lt, t) in enumerate((('a', 'Agreement on referral decisions'), ('b', 'Agreement on score rankings'))):
    axes[row][0].annotate(f'{lt}  {t}', xy=(0, 1), xycoords='axes fraction', xytext=(-205, 30), textcoords='offset points', fontsize=10, weight='bold', ha='left')
fig.legend(handles=[Line2D([], [], color=GREEN, ls='-', lw=1.4, label='Relative rule: 95th percentile of agreement among systems known to differ in design')],
           loc='lower left', bbox_to_anchor=(0.01, -0.01), fontsize=8)
title(fig, 'Figure 5. Related systems give themselves away on local unlabelled images')
fig.tight_layout(rect=(0, 0.03, 1, 0.97)); fig.subplots_adjust(hspace=0.55); save(fig, 'Fig5_siblings'); print(json.dumps(DET, indent=1))

saved Fig5_siblings
{
 "EyePACS": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.5979,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6886,
   "siblings_flagged": 1.0
  }
 },
 "DDR": {
  "kappa": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.6478,
   "siblings_flagged": 1.0
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 1.0,
   "relative_threshold": 0.8423,
   "siblings_flagged": 1.0
  }
 },
 "BRSET": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.879,
   "relative_threshold": 0.6721,
   "siblings_flagged": 0.625
  },
  "rank_corr": {
   "auc_siblings_vs_different_design": 0.998,
   "relative_threshold": 0.659,
   "siblings_flagged": 1.0
  }
 },
 "HIBA": {
  "kappa": {
   "auc_siblings_vs_different_design": 0.947,
   "relative_threshold": 0.6856,
   "siblings_flagged": 0.833
  },
  "rank_corr": {
   "auc_siblings_vs_different_

In [12]:
# Supplementary Figure 13: pooled training versus pairing (notebook 26; plan dated before training)
ARMS = [('single_E', 'EyePACS-only single', '#9ecae1'), ('single_D', 'DDR-only single', '#fdae6b'), ('single_P', 'Pooled single', '#a1d99b'),
        ('pair_mixed_E_D', 'EyePACS-trained + DDR-trained pair', ORANGE), ('pair_pooled', 'Pooled pair', GREEN), ('pair_E', 'EyePACS-only pair', BLUE)]
SETS = [('BRSET | severe', 'BRSET (external to every model)'), ('EyePACS fresh | severe', 'EyePACS test set'), ('DDR test | severe', 'DDR official test file')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for a_, (key, ttl), lt in zip(axes, SETS, 'abc'):
    r = POOL[key]['missed_per_10k']
    for i, (k, lab, c) in enumerate(ARMS):
        m, lo, hi = r[k]; a_.bar(i, m, 0.75, color=c); a_.errorbar(i, m, yerr=[[max(0, m - lo)], [max(0, hi - m)]], color='#333333', lw=1, capsize=2.5)
    a_.set_xticks(range(len(ARMS))); a_.set_xticklabels([a[1] for a in ARMS], rotation=40, ha='right', fontsize=7.4); a_.set_ylabel('Missed severe DR per 10,000 images')
    panel(a_, lt, ttl)
v = POOL['verdicts_brset_severe']
axes[0].text(0, -0.95, 'Pre-specified (BRSET): ' + '; '.join(f'{k.split()[0]} {s}' for k, s in v.items()), transform=axes[0].transAxes, fontsize=7.6, color='#444444')
title(fig, 'Supplementary Figure 13. Pooled training versus pairing (same referral share; singles averaged over 12 models; pairs combine two different designs; 95% patient bootstrap)')
fig.tight_layout(); save(fig, 'SuppFig13_pooled_training')

saved SuppFig13_pooled_training


In [13]:
# Numbers in the figures next to the manuscript's; aggregate record
for k, v in CHECK.items(): print(k, '->', json.dumps(v, default=float))
json.dump({'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'status': 'supplementary figures after review cycle 1; aggregates only',
           'checks': CHECK, 'sweeps': {f'{s} | {o}': {k: {'grid': (GRID_R if s in ('EyePACS', 'DDR') else GRID_S).tolist(), 'estimate_lo_hi': v.round(4).tolist()} for k, v in d.items()} for (s, o), d in SW.items()},
           'rescue_by_grade': F3S, 'sibling_detector': DET}, open(f'{OUT}/supp_figures_c1_data.json', 'w'), indent=1, default=float)
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 05_figures/manuscript_final: SuppFig1 to SuppFig7, Fig5, SuppFig13, supp_figures_c1_data.json; previous versions renamed SUPERSEDED_')

SuppFig1 DDR strongest partner range -> {"figure": "2% to 51%", "manuscript": "26% to 85%"}
SuppFig2 skin all pairs -> {"figure": {"HIBA melanoma": "4.3% to 43.5%", "HIBA any skin cancer": "0.9% to 56.8%", "PAD any skin cancer": "0.6% to 27.7%"}, "derm_referral_share_results (27 Sep)": "HIBA melanoma 4.3% to 43.5%; HIBA any 0.9% to 56.9%; PAD any 0.6% to 27.6%"}
SuppFig3 BRSET matched-volume rescue by grade -> {"figure": {"moderate": 0.057, "severe NPDR": 0.333, "PDR": 0.097}, "brset_results_v6 B5": {"grade_2": 0.057, "grade_3": 0.3333, "grade_4": 0.0966}}
SuppFig4 severe DR -> {"seed_siblings": 11.65, "same_arch_diff_init_pooled": 8.09, "different_architecture": 7.69}
SuppFig5 degradation -> {"figure": 0.406, "interval": [0.178, 0.597], "manuscript": "0.41 (0.18 to 0.60)"}
SuppFig7 patient split, strongest / best possible -> {"figure": {"EyePACS": {"25": [40.2, 30.0], "50": [52.6, 38.8], "100": [74.9, 55.3], "200": [78.6, 55.5]}, "DDR": {"25": [19.1, 17.2], "50": [36.0, 32.4], "100": 